# EM Algorithm for DFM Estimation

**Goal:** Implement the EM algorithm from scratch.

**Time:** 15 minutes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulate data
np.random.seed(42)
T = 200
true_Q = 0.5
true_H = 1.0

states = np.cumsum(np.random.randn(T) * np.sqrt(true_Q))
y = states + np.random.randn(T) * np.sqrt(true_H)

print(f"Data generated: T={T}, true Q={true_Q}, true H={true_H}")

## Kalman Smoother (E-step)

In [ ]:
def kalman_smoother(y, Q, H):
    """Kalman filter + RTS smoother."""
    T = len(y)
    
    # Forward pass (filter)
    a_filt = np.zeros(T)
    P_filt = np.zeros(T)
    a_pred = np.zeros(T)
    P_pred = np.zeros(T)
    
    a, P = 0.0, 10.0
    
    for t in range(T):
        # Predict
        a_pred[t] = a
        P_pred[t] = P + Q
        
        # Update
        v = y[t] - a_pred[t]
        F = P_pred[t] + H
        K = P_pred[t] / F
        
        a = a_pred[t] + K * v
        P = P_pred[t] - K * F * K
        
        a_filt[t] = a
        P_filt[t] = P
    
    # Backward pass (smoother)
    a_smooth = np.zeros(T)
    P_smooth = np.zeros(T)
    
    a_smooth[-1] = a_filt[-1]
    P_smooth[-1] = P_filt[-1]
    
    for t in range(T-2, -1, -1):
        J = P_filt[t] / P_pred[t+1]
        a_smooth[t] = a_filt[t] + J * (a_smooth[t+1] - a_pred[t+1])
        P_smooth[t] = P_filt[t] + J * (P_smooth[t+1] - P_pred[t+1]) * J
    
    return a_smooth, P_smooth

# Test
a_smooth, P_smooth = kalman_smoother(y, true_Q, true_H)
print(f"Smoother computed. Mean state: {a_smooth.mean():.3f}")

## EM Algorithm

In [ ]:
def em_algorithm(y, max_iter=50, tol=1e-4):
    """EM algorithm for local level model."""
    # Initialize
    Q = 0.5
    H = 1.0
    
    loglik_hist = []
    Q_hist = [Q]
    H_hist = [H]
    
    for iteration in range(max_iter):
        # E-step: Run smoother
        a_smooth, P_smooth = kalman_smoother(y, Q, H)
        
        # M-step: Update parameters
        # Q update
        Q_new = np.mean((a_smooth[1:] - a_smooth[:-1])**2 + 
                        P_smooth[1:] + P_smooth[:-1])
        
        # H update
        H_new = np.mean((y - a_smooth)**2 + P_smooth)
        
        # Compute log-likelihood
        loglik = 0.0
        a, P = 0.0, 10.0
        for t in range(len(y)):
            a_pred = a
            P_pred = P + Q_new
            v = y[t] - a_pred
            F = P_pred + H_new
            K = P_pred / F
            a = a_pred + K * v
            P = P_pred - K * F * K
            loglik += -0.5 * (np.log(2*np.pi) + np.log(F) + v**2/F)
        
        loglik_hist.append(loglik)
        Q_hist.append(Q_new)
        H_hist.append(H_new)
        
        # Check convergence
        if iteration > 0 and abs(loglik - loglik_hist[-2]) < tol:
            print(f"Converged after {iteration+1} iterations")
            break
        
        Q, H = Q_new, H_new
    
    return Q, H, loglik_hist, Q_hist, H_hist

# Run EM
Q_hat, H_hat, loglik_hist, Q_hist, H_hist = em_algorithm(y)

print(f"\nEM Estimates:")
print(f"  Q: {Q_hat:.3f} (true: {true_Q:.3f})")
print(f"  H: {H_hat:.3f} (true: {true_H:.3f})")
print(f"\nFinal log-likelihood: {loglik_hist[-1]:.2f}")

## Visualize Convergence

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# Log-likelihood
axes[0].plot(loglik_hist, 'o-', linewidth=2)
axes[0].set_title('EM Algorithm: Log-Likelihood Convergence')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Log-Likelihood')
axes[0].grid(True, alpha=0.3)

# Q convergence
axes[1].plot(Q_hist, 'o-', linewidth=2, label='Q estimate')
axes[1].axhline(true_Q, color='red', linestyle='--', linewidth=2, label='True Q')
axes[1].set_title('Q Parameter Convergence')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Q')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# H convergence
axes[2].plot(H_hist, 'o-', linewidth=2, label='H estimate')
axes[2].axhline(true_H, color='red', linestyle='--', linewidth=2, label='True H')
axes[2].set_title('H Parameter Convergence')
axes[2].set_xlabel('Iteration')
axes[2].set_ylabel('H')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nEM guarantees likelihood increases each iteration!")
print("Notice smooth, monotonic convergence to true parameters.")